# 05. 영상 시퀀스 파이프라인 (좌표 -> 리샘플링 -> 각도 시퀀스 -> 경기 텐서)

> **핵심 로직은 `video_sequence_features.py`로 모듈화**, 이 노트북은 **import해서 호출만** 한다.
> (04_video_pipeline.ipynb / feature_aggregator.py와 동일 패턴 -- 노트북=실행/기록, .py=재사용 함수)

04번(정적 릴리스-1프레임 → 9-stat 집계)과 다른 점: 이 노트북은 **투구 장면 전체 프레임**을
고정 길이로 리샘플링해 시계열 그대로 남긴다. 03_skeleton.ipynb를 `EXTRACT_SEQUENCE=True`로
실행해 만든 `batch_slot*_seq.parquet`가 입력이다(04번이 쓰는 `*_coords.csv`와는 별도 산출물).

| 단계 | 함수 (video_sequence_features.py) | 입력 -> 출력 |
|------|-----------------------------------|-------------|
| A. 시퀀스 합치기 | `merge_sequences()` | `batch_slot*_seq.parquet` -> 프레임 단위 long table + 경기정보 |
| B. 투구별 각도 시퀀스 | `build_pitch_sequences()` | 프레임 -> 좌투 미러링 -> 고정 T 리샘플링 -> 각도 9종 시퀀스 |
| C. 경기 텐서 조립 | `build_game_tensor()` | 투구별 시퀀스 -> (경기, 최대15투구, T, 9각도) 텐서 + mask |
| **전체 일괄** | `build_and_save()` | 위 3단계 -> `video_seq_pitch{N}_T{T}.npz` + 메타 parquet |

⚠ **투구 축은 순서가 없다**: 02_video_collect.ipynb가 모으는 play_id는 실제 pitch_number를
보존하지 않는다(README "다음 단계" 참고). 그래서 경기 텐서의 투구 축(최대 15개)은
set(순서 무관)으로 취급해야 하며, 17_video_sequence_experiment.ipynb의 모델도 이 축엔
순서를 학습하는 레이어 대신 masked mean/std pooling을 쓴다. 반대로 투구 1개 내부의
프레임 축은 실제 시간 순서이므로 CNN을 그대로 써도 된다.

재실행: 영상이 더 추출되면(03번을 EXTRACT_SEQUENCE=True로 이어서 돌리면) 이 노트북만
Run All. 결과는 `17_video_sequence_experiment.ipynb`가 읽는다.


## 0. 환경·경로 설정

In [ ]:
import os, sys

# 환경 자동 감지 (Colab / 로컬) -- 04_video_pipeline.ipynb와 동일 규칙
IN_COLAB = os.path.exists('/content')
if IN_COLAB:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')
    BASE       = '/content/drive/MyDrive/MLB_pitcher'
    MODULE_DIR = BASE                       # video_sequence_features.py 위치 (MLB_pitcher 루트, 서브폴더 없음)
    OUTPUT_DIR = f'{BASE}/output'           # batch_slot*_seq.parquet (03_skeleton.ipynb 출력)
    PLAY_IDS   = f'{BASE}/data/play_ids_sample.csv'
    FEAT_DIR   = f'{BASE}/data/4_features'  # 출력 npz/parquet
else:
    BASE       = r'c:\Users\suyou\OneDrive\Desktop\ASAC\PROJECT\투수 컨디션 예측'
    MODULE_DIR = os.path.join(BASE, '2_video')
    OUTPUT_DIR = os.path.join(BASE, '0_data', 'output')
    PLAY_IDS   = os.path.join(BASE, '0_data', 'data', 'play_ids_sample.csv')
    FEAT_DIR   = os.path.join(BASE, '0_data', '4_features')

# video_sequence_features.py 를 import 경로에 추가
sys.path.insert(0, MODULE_DIR)
import video_sequence_features as vsf
import importlib; importlib.reload(vsf)   # 모듈 수정 시 반영

SLOTS = [0, 1, 2, 3, 4]   # 시즌(2021~2025)별 슬롯 -- video_features.py와 동일 규약
print(f'환경: {"Colab" if IN_COLAB else "로컬"} | 모듈: {MODULE_DIR}')
print(f'시퀀스 원본(*_seq.parquet): {OUTPUT_DIR} | 슬롯: {SLOTS}')
print(f'T(프레임 리샘플 길이)={vsf.T_RESAMPLE}  max_pitches={vsf.MAX_PITCHES}')


## 1. 전체 일괄 실행 (`build_and_save`)

`merge_sequences -> build_pitch_sequences -> build_game_tensor`를 한 번에.
결과는 `video_seq_pitch{max_pitches}_T{T}.npz`(X, X_rel, mask)와 메타 parquet.


In [ ]:
X, X_rel, mask, meta = vsf.build_and_save(
    output_dir=OUTPUT_DIR,
    play_ids_csv=PLAY_IDS,
    out_dir=FEAT_DIR,
    slots=SLOTS,
)
print(f'\n완료. X{X.shape} -> {FEAT_DIR}')
print(f'경기 수: {len(meta):,} | 시즌별: {meta["season"].value_counts().sort_index().to_dict()}')


## 2. (선택) 단계별 실행 -- 중간 결과 확인용

`build_and_save` 대신 단계를 나눠 돌리며 중간 산출물을 보고 싶을 때.


In [ ]:
# A. 시퀀스 배치 파일 미리보기 (전체를 합치지 않음 — OOM 방지)
# ⚠ merge_sequences()는 전체를 하나의 DataFrame으로 합쳐서 대규모 데이터에서 메모리 위험이
#   있다(모듈 docstring 참고). 여기서는 배치 1개만 살짝 열어 컬럼/shape만 확인한다.
_first_batch = next(vsf.iter_sequence_batches(OUTPUT_DIR, PLAY_IDS, slots=SLOTS))
print(f'A. 배치 샘플 1개: {_first_batch.shape}  |  컬럼: {list(_first_batch.columns)[:8]}...')
print(f'   이 배치에 포함된 투구(영상) 수: {_first_batch["video_name"].nunique()}')
del _first_batch

# B. 투구별 각도 시퀀스 — 배치 단위 스트리밍 처리(build_and_save가 내부적으로 쓰는 것과 동일 함수)
pitch_seqs = vsf.build_pitch_sequences_streaming(OUTPUT_DIR, PLAY_IDS, slots=SLOTS, T=vsf.T_RESAMPLE)
sample_key = next(iter(pitch_seqs))
print(f'B. 투구별 시퀀스: {len(pitch_seqs):,}개 | 샘플({sample_key}) angles.shape='
      f'{pitch_seqs[sample_key]["angles"].shape}')

# C. 경기 단위 텐서 조립 (투구 축 = 순서 무관 set, 패딩 포함)
X, X_rel, mask, meta = vsf.build_game_tensor(pitch_seqs, max_pitches=vsf.MAX_PITCHES, T=vsf.T_RESAMPLE)
print(f'\nC. 경기 텐서: X{X.shape}  |  경기당 평균 투구 수: {meta["n_pitches_used"].mean():.1f}')
meta.head(3)
